# Why per-step reward centering starves the value function

**Question.** In a differential / average-reward return $R_t=\sum_{t'\ge t}\gamma^{t'-t}\big(r_{t'}-\mathbb{E}[r_{t'}]\big)$,
what happens to the value function $V_t=\mathbb{E}[R_t\mid F_t]$ under different choices of the baseline
$\mathbb{E}[r_{t'}]$? In particular:

1. $\mathbb{E}[r_{t'}\mid F_{t'}]$ — the **contemporaneous** one-step predictor (formed with the current observation),
2. $\mathbb{E}[r_{t'}\mid F_{t'-1}]$ — the **past-conditioned** predictor (formed *before* the current observation),
3. a **global average reward** $\rho$ (a scalar, not a per-step conditional mean).

Notation: $F_t$ is everything observed through step $t$ (so the observation $o_t\in F_t$), but not the reward
$r_t$ itself, which the value must predict.

## The argument (tower property)

The residuals $r_{t'}-\mathbb{E}[r_{t'}\mid F_{t'-1}]$ are the **innovations** of the reward-prediction
process: a martingale-difference sequence. A *future* innovation is unpredictable from anything known now,
because if you could predict it, so could the one-step-ahead predictor at $t'-1$ (which knows strictly more
than you do at $t$) — contradicting it being a surprise. Formally, for any $t' > t$, since
$F_t\subseteq F_{t'-1}$,

$$\mathbb{E}\big[\,r_{t'}-\mathbb{E}[r_{t'}\mid F_{t'-1}]\ \big|\ F_t\,\big]
=\mathbb{E}[r_{t'}\mid F_t]-\mathbb{E}\big[\mathbb{E}[r_{t'}\mid F_{t'-1}]\mid F_t\big]=0.$$

So **every future term cancels**, regardless of the baseline's timing. Only the *current* term ($t'=t$) can
survive, and only if the value sees more than the baseline did:

$$\mathbb{E}\big[\,r_t-\mathbb{E}[r_t\mid \cdot]\ \big|\ F_t\,\big]=\mathbb{E}[r_t\mid F_t]-\mathbb{E}[r_t\mid \cdot].$$

**Case 1 — contemporaneous baseline** $\mathbb{E}[r_t\mid F_t]$: the current term is
$\mathbb{E}[r_t\mid F_t]-\mathbb{E}[r_t\mid F_t]=0$. Nothing survives — $V_t\equiv 0$. The value has no target.

**Case 2 — past-conditioned baseline** $\mathbb{E}[r_t\mid F_{t-1}]$: the current term is
$\mathbb{E}[r_t\mid F_t]-\mathbb{E}[r_t\mid F_{t-1}]$ — the **one-step reward-prediction update** from observing
$o_t$. Nonzero, so $V$ no longer collapses, but it is a *myopic* one-step signal: the multi-step part of the
return is still zero-mean noise. (And it survives only because the critic conditions on $o_t$ while the
baseline does not — and only if the return includes the current step; a "future-only" value goes back to $0$.)

**Case 3 — global average $\rho$**: $\rho$ is *not* a one-step-ahead expectation of $r_{t'}$, so the tower
argument doesn't apply. $r_{t'}-\rho$ has nonzero conditional mean at every step (the state's expected reward
minus the global rate), the future terms carry signal, and $V_t$ is a **genuine long-horizon differential
value**. This is why centering by the slow `exp_filtered_reward_rate` works while a per-step predictor does not.

## A concrete numerical check

A minimal environment that has real temporal structure: a latent "patch quality" $s_t$ following an AR(1)
process, observed as $o_t=s_t$, with reward $r_t=a\,s_t+\text{noise}$.

- $\mathbb{E}[r_t\mid F_t]=a\,s_t$ (uses $o_t$),
- $\mathbb{E}[r_t\mid F_{t-1}]=a\,\phi\,s_{t-1}$ (AR(1) one-step-ahead, before $o_t$),
- $\rho=\mathbb{E}[r]$ (a scalar).

For each baseline we build the Monte-Carlo return $G_t$, estimate the value $V_t=\mathbb{E}[G_t\mid F_t]$ by
regressing $G_t$ on $F_t=(s_t,s_{t-1})$, and report `std(value)` (how much target signal there is) and the
fraction of return variance that is predictable.

In [1]:
import numpy as np
rng = np.random.default_rng(0)

T = 200_000
phi, a, sig_eps, sig_r, gamma = 0.9, 1.0, 1.0, 0.5, 0.95   # AR(1) quality, reward loading, noises, discount

# latent "patch quality" AR(1):  s_t = phi s_{t-1} + eps_t ,  observed as o_t = s_t
eps = rng.normal(0, sig_eps, T)
s = np.zeros(T)
for t in range(1, T):
    s[t] = phi * s[t-1] + eps[t]
r = a * s + rng.normal(0, sig_r, T)          # reward r_t = a*s_t + noise  (o_t reveals s_t)
s_prev = np.concatenate([[0.0], s[:-1]])

# three choices of the baseline E[r_t] to subtract:
E_contemp = a * s                             # E[r_t | F_t]      -- uses the current obs o_t
E_past    = a * phi * s_prev                   # E[r_t | F_{t-1}]  -- formed BEFORE o_t
rho       = r.mean()                           # global average reward (a scalar)

def disc_return(x, g):                         # G_t = sum_{i>=0} g^i x_{t+i}
    G = np.zeros_like(x); acc = 0.0
    for t in range(len(x) - 1, -1, -1):
        acc = x[t] + g * acc; G[t] = acc
    return G

# The value is E[G_t | F_t]; F_t is summarized by (s_t, s_{t-1}). Estimate it by linear regression.
X = np.column_stack([np.ones(T), s, s_prev])
def analyze(center, name):
    G = disc_return(r - center, gamma)
    beta, *_ = np.linalg.lstsq(X, G, rcond=None)
    V = X @ beta                               # fitted value = E[G_t | F_t]
    frac = 1 - np.var(G - V) / np.var(G)        # fraction of return variance that is predictable
    tracks = '(value magnitude ~ 0  ->  no target)'
    if V.std() > 0.1:      # only report what V tracks when it has real magnitude
        c_s   = np.corrcoef(V, s)[0, 1]
        c_inn = np.corrcoef(V, s - phi * s_prev)[0, 1]      # innovation eps_t
        tracks = f'corr(V, s_t)={c_s:+.2f}   corr(V, innovation)={c_inn:+.2f}'
    print(f'{name:30s} std(value)={V.std():7.3f}   predictable frac={frac:5.3f}   {tracks}')

print('baseline  E[r_t] =')
analyze(E_contemp, '  E[r_t | F_t]    (contemp)')
analyze(E_past,    '  E[r_t | F_{t-1}]  (past)')
analyze(rho,       '  rho = mean(r)   (global)')

print(f'\nanalytic std(value):  contemp ~ 0'
      f'  |  past = a*sig_eps = {a*sig_eps:.2f}'
      f'  |  global = a*sqrt(Var s)/(1-gamma*phi) = {a*np.sqrt(sig_eps**2/(1-phi**2))/(1-gamma*phi):.2f}')

baseline  E[r_t] =
  E[r_t | F_t]    (contemp)    std(value)=  0.028   predictable frac=0.000   (value magnitude ~ 0  ->  no target)
  E[r_t | F_{t-1}]  (past)     std(value)=  0.994   predictable frac=0.078   corr(V, s_t)=+0.43   corr(V, innovation)=+1.00
  rho = mean(r)   (global)     std(value)= 15.937   predictable frac=0.364   corr(V, s_t)=+1.00   corr(V, innovation)=+0.43

analytic std(value):  contemp ~ 0  |  past = a*sig_eps = 1.00  |  global = a*sqrt(Var s)/(1-gamma*phi) = 15.82


## Reading the numbers

- **Contemporaneous** `E[r_t|F_t]`: `std(value) ≈ 0`, predictable fraction `≈ 0`. The value has **no target** —
  once you subtract the best current-step prediction, all that's left are unpredictable innovations.
- **Past** `E[r_t|F_{t-1}]`: `std(value) ≈ a·σ_ε`, and the value is **perfectly correlated with the current
  innovation $\varepsilon_t=s_t-\phi s_{t-1}$** (and only weakly with $s_t$). It escaped the exact-zero
  collapse, but only captures **one step** of surprise — a myopic signal, not an accumulating value.
- **Global $\rho$**: `std(value)` is an order of magnitude larger (matching $a\sqrt{\mathrm{Var}\,s}/(1-\gamma\phi)$),
  a large predictable fraction, and the value is **perfectly correlated with $s_t$** — a real long-horizon
  differential value that integrates future reward.

**Takeaway.** No per-step conditional mean can serve as the centering for a value that accumulates: whatever
information it conditions on, the tower property annihilates the future, leaving at most one step of signal.
For a genuine differential value you must subtract something that is *not* a one-step-ahead expectation of the
reward — a global / slowly-varying average reward $\rho$. (In the training scripts, that's why the slow
`exp_filtered_reward_rate` centering gives the critic a real target, whereas an actor reward predictor used as
$\mathbb{E}[r_t]$ would leave it starved.)